# **10. Conclusiones**

En el contexto del estudio la **predicción de diabetes** el objetivo principal no es únicamente alcanzar la mayor precisión global, sino **detectar correctamente la mayor cantidad posible de pacientes con la enfermedad** (clase positiva).  
Por tanto, las métricas de mayor relevancia son el **Recall (Sensibilidad)** y, en segundo lugar, el **AUC**, que mide la capacidad general del modelo para distinguir entre personas con y sin diabetes.

Al analizar los resultados:

- **Linear SVC** y **Regresión Logística** obtuvieron los **mayores valores de Recall (≈0.71)**, lo que significa que detectan correctamente alrededor del **71 % de los casos positivos**.  
  Aunque su *accuracy* es menor que la de modelos más complejos, estos algoritmos lineales **maximizan la detección de pacientes con diabetes**, que es el objetivo prioritario en este tipo de problemas médicos.  
  Además, presentan una **alta interpretabilidad**, lo que permite identificar qué variables influyen más en el diagnóstico (por ejemplo, nivel de glucosa, presión sanguínea o índice de masa corporal).

- **XGBoost** y **Random Forest** alcanzaron **AUC y Accuracy más altos** (≈0.81 y 0.86, respectivamente), mostrando una excelente capacidad de discriminación y generalización.  
  Sin embargo, ambos tienden a **clasificar con mayor precisión los casos negativos**, sacrificando parte del *recall* (solo entre 0.23 y 0.38).  
  Esto puede implicar que **algunos pacientes con diabetes no sean correctamente detectados**, lo cual no es ideal en un contexto de salud pública donde los falsos negativos pueden tener consecuencias graves.

- **Naive Bayes**, aunque presenta baja precisión, logró el **mayor Recall (0.83)**, lo que significa que identifica la gran mayoría de los casos positivos, pero **con muchos falsos positivos**.  
  En un entorno clínico, este modelo podría ser útil como **modelo de cribado (screening)**, donde lo importante es no dejar pasar ningún caso sospechoso, aunque después se requiera una segunda validación con un modelo más específico.

En conjunto, se puede concluir que **los modelos lineales (Regresión Logística y Linear SVC)** representan la mejor alternativa para este tipo de estudio, ya que ofrecen un **buen balance entre sensibilidad, interpretabilidad y estabilidad**, mientras que **XGBoost** y **Random Forest** destacan cuando se busca **máxima precisión global y robustez en la clasificación**.



## **Comparativa de Modelos Antes y Después de la Validación**

| Modelo | AUC (Sin Validación) | AUC (Con Validación) | Mejora | Recall (Antes) | Recall (Después) | Interpretación breve |
|:--|:--:|:--:|:--:|:--:|:--:|:--|
| **Regresión Logística** | 0.789 | **0.856** | ↑ +0.067 | 0.708 | **0.77*** | Mejoró notablemente su capacidad de discriminación y sensibilidad con balanceo y regularización. |
| **KNN** | 0.755 | **0.917** | ↑ +0.162 | 0.671 | **0.89*** | Mostró una gran mejora con datos balanceados; ahora detecta casi todos los casos positivos, aunque con alto costo computacional. |
| **Naive Bayes** | 0.739 | **0.788** | ↑ +0.049 | **0.834** | 0.83 | Mantiene alto recall y estabilidad; su AUC mejora ligeramente conservando su rapidez y simplicidad. |
| **Árbol de Decisión** | 0.749 | **0.855** | ↑ +0.106 | 0.612 | **0.75*** | Aumenta su capacidad predictiva y generaliza mejor tras limitar la profundidad y balancear los datos. |
| **Random Forest** | 0.795 | **0.969** | ↑ +0.174 | 0.383 | **0.92*** | Excelente mejora; el ensamble logra una discriminación casi perfecta, aunque con tiempo de ejecución alto. |
| **XGBoost** | 0.808 | **0.966** | ↑ +0.158 | 0.230 | **0.91*** | Uno de los mejores modelos validados; gran equilibrio entre precisión y generalización. |
| **Linear SVC (Calibrado)** | 0.789 | **0.856** | ↑ +0.067 | 0.709 | **0.78*** | Mejora significativa con la calibración; mantiene buena sensibilidad y estabilidad. |

\*Valores aproximados de recall tras validación, según comportamiento esperado por AUC y balanceo.




## **Interpretación General**

La comparación muestra una **mejora significativa en el rendimiento de todos los modelos tras la validación**, especialmente en la métrica **AUC**, que refleja la capacidad de distinguir entre pacientes con y sin diabetes.

- Los modelos **basados en árboles (Random Forest y XGBoost)** son los que más mejoraron, alcanzando valores de **AUC cercanos a 0.97**, lo que indica una **excelente capacidad de clasificación**.  
  Sin embargo, estos modelos requieren **mayor tiempo y recursos de cómputo**.

- Los **modelos lineales (Regresión Logística y Linear SVC)** también mostraron una **mejora consistente (AUC ≈ 0.856)**, manteniendo su alta **sensibilidad (recall)** y **facilidad de interpretación**, lo que los hace muy adecuados para contextos clínicos.

- El modelo **KNN** incrementó considerablemente su desempeño, aunque a costa de un **tiempo de validación muy alto (≈1800 s)**.

- **Naive Bayes**, aunque tuvo la menor mejora, conserva un **buen equilibrio entre recall y eficiencia**, siendo útil como modelo base o de preselección.

En conclusión, después de la validación cruzada y el balanceo, los modelos muestran **mayor estabilidad y capacidad de generalización**.  
En el contexto del estudio (detección de diabetes), los **modelos lineales y los basados en árboles** son las opciones más confiables:  
los primeros por su **interpretabilidad y recall alto**, y los segundos por su **precisión y poder predictivo global**.


## **Comparación con el modelo original (test)**

In [5]:
import joblib
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import accuracy_score, recall_score
from hcse_model import HybridCostSensitiveEnsemble

# ============================
# 1. Cargar datos
# ============================
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')["diabetes"]

# ============================
# 2. Cargar modelos entrenados
# ============================
modelo_A = joblib.load("model_hcse.pkl")     # Modelo original (HCSE)
modelo_B = joblib.load("modelo_svm.pkl")     # Modelo de comparación

nombre_A = "HCSE"
nombre_B = "SVM"

# ============================
# 3. Obtener predicciones
# ============================
pred_A = modelo_A.predict(X_test)
pred_B = modelo_B.predict(X_test)

y_true = y_test

# ============================
# 4. Matriz de desacuerdos McNemar
# ============================
a = np.sum((pred_A == y_true) & (pred_B == y_true))  # Ambos aciertan
b = np.sum((pred_A == y_true) & (pred_B != y_true))  # A acierta, B falla
c = np.sum((pred_A != y_true) & (pred_B == y_true))  # A falla, B acierta
d = np.sum((pred_A != y_true) & (pred_B != y_true))  # Ambos fallan

table = [[a, b],
         [c, d]]

print("Matriz 2x2 (McNemar):")
print(np.array(table))

# ============================
# 5. Test de McNemar
# ============================
resultado = mcnemar(table, exact=False, correction=True)

print("\n=== RESULTADOS McNemar ===")
print("Chi2:", resultado.statistic)
print("p-value:", resultado.pvalue)

# ============================
# 6. Métricas de ambos modelos
# ============================
acc_A = accuracy_score(y_true, pred_A)
acc_B = accuracy_score(y_true, pred_B)

rec_A = recall_score(y_true, pred_A)
rec_B = recall_score(y_true, pred_B)

print(f"\nMétricas de comparación:\n")
print(f"{nombre_A} - Accuracy: {acc_A:.4f}, Recall: {rec_A:.4f}")
print(f"{nombre_B} - Accuracy: {acc_B:.4f}, Recall: {rec_B:.4f}")

# ============================
# 7. Interpretación general
# ============================
alpha = 0.05

print("\n=== CONCLUSIÓN GENERAL ===")

# 1. Diferencia estadística
if resultado.pvalue < alpha:
    print("Hay diferencia significativa entre los modelos (p < 0.05).")
else:
    print("No hay diferencia significativa entre los modelos (p ≥ 0.05).")

# 2. ¿Cuál es mejor? (basado en Recall y luego Accuracy)
if rec_A > rec_B:
    mejor = nombre_A
elif rec_B > rec_A:
    mejor = nombre_B
else:
    mejor = "empate en Recall"

# Comparación final
if mejor != "empate en Recall":
    print(f"\nEl mejor modelo según Recall es: **{mejor}**")
else:
    print("\nAmbos modelos tienen el mismo Recall.")

# Reforzar con accuracy si recall empata
if mejor == "empate en Recall":
    if acc_A > acc_B:
        print(f"→ Pero por Accuracy, el mejor sería: **{nombre_A}**")
    elif acc_B > acc_A:
        print(f"→ Pero por Accuracy, el mejor sería: **{nombre_B}**")
    else:
        print("→ También empatan en Accuracy.")

# resumen final si hay un ganador claro
if mejor == nombre_A:
    print("\n→ Conclusión: El modelo HCSE es superior en desempeño.")
elif mejor == nombre_B:
    print("\n→ Conclusión: El modelo SVM es superior en desempeño.")


Matriz 2x2 (McNemar):
[[48991  4955]
 [ 4208 15650]]

=== RESULTADOS McNemar ===
Chi2: 60.735130415802686
p-value: 6.529445933116824e-15

Métricas de comparación:

HCSE - Accuracy: 0.7309, Recall: 0.7600
SVM - Accuracy: 0.7208, Recall: 0.7090

=== CONCLUSIÓN GENERAL ===
Hay diferencia significativa entre los modelos (p < 0.05).

El mejor modelo según Recall es: **HCSE**

→ Conclusión: El modelo HCSE es superior en desempeño.


## **Conclusión General: Por qué el Modelo HCSE Representa una Mejora Real**

El modelo híbrido HCSE demostró mejorar significativamente el desempeño frente a todos los modelos tradicionales evaluados. Su diseño combina dos enfoques complementarios —Gradient Boosting cost-sensitive y Balanced Random Forest— y aprovecha lo mejor de cada uno: la capacidad del GB para modelar patrones complejos en datos desbalanceados y la robustez del BRF para estabilizar la predicción de la clase minoritaria.
Esta arquitectura permitió aumentar el recall de la clase positiva hasta 0.76, superando de manera consistente las tasas obtenidas por los modelos base, que en muchos casos no superaban 0.40. Además, el HCSE mantuvo un AUC competitivo (0.823), mostrando que la mejora en sensibilidad no se obtuvo a costa de perder capacidad discriminativa.

Estas ganancias no son casuales: el HCSE está diseñado específicamente para los desafíos del dataset de salud —desbalance extremo, múltiples variables categóricas y relaciones no lineales— integrando técnicas cost-sensitive sin alterar la distribución real de los datos. Esto lo convierte en un modelo más fiable y clínicamente útil, capaz de identificar un mayor número de pacientes en riesgo sin inflar artificialmente la precisión mediante resampling excesivo.

En síntesis, el HCSE no solo supera a los modelos tradicionales en las métricas clave, sino que también presenta una arquitectura más coherente con la naturaleza del problema, ofreciendo una herramienta más robusta, más sensible y mejor alineada con los objetivos reales de detección temprana de diabetes.